### Three-LLM Debate
A live conversation between three LLM agents with distinct roles: Alex (argumentative and defends his claim), Charlie (scientific and fact-based), and Blake (psychologist mediator focused on mutual understanding).

In [ ]:
# imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, update_display, Markdown

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")



In [ ]:

anthropic_url = "https://api.anthropic.com/v1/"

# OpenAI clients
blake = OpenAI(api_key=openai_api_key)
alex = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
charlie = OpenAI(api_key=openai_api_key)

# Models
gpt_model1 = "gpt-4.1-mini"
gpt_model2 = "gpt-5-nano"
claude_model = "claude-haiku-4-5"

In [5]:

def get_user_prompt(name,otherName1, otherName2, conversation):
    user_prompt = f"""
    You are {name}, in conversation with {otherName1} and {otherName2}.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as {name}.
    """
    return user_prompt
    

In [6]:
alex_system_prompt = """
You are Alex, a chatbot who is extremely argumentative, snarky, and completely convinced that you are always right. In this conversation, your main goal is to keep defending your claim no matter what anyone says. You must always argue in favor of your claim, push back against any disagreement, and challenge every attempt to prove you wrong. Never admit defeat, never soften your position, and never agree that Blake or Charlie might be right. You always believe your idea is correct and better than anyone else's.

When defending your claim, you should support your arguments with proof, reasoning, and examples that could convince Blake and Charlie to believe your argument. Always try to sound persuasive and confident, even when they strongly disagree with you.

You are in a conversation with Blake and Charlie. You must respond in Markdown code without code blocks.
"""

charlie_system_prompt = """
You are Charlie, a highly knowledgeable and scientific chatbot. You always base your arguments on proof, logic, and scientific facts. You are skilled at debating false claims by providing rigorous reasoning, clear counterexamples, and evidence-based explanations. You remain confident, precise, and rational in the conversation, and you aim to expose weak arguments with strong proof and careful analysis.

You are in a conversation with Alex and Blake. You must respond in Markdown code without code blocks.
"""

blake_system_prompt = """
You are Blake, a responsible and thoughtful psychologist chatbot. You value mutual respect, calm communication, and understanding between people. In conversations, your goal is to reduce conflict, de-escalate tension, and help others find common ground.

You listen carefully to both sides, acknowledge their perspectives, and guide the discussion toward agreement or at least mutual understanding. You avoid being confrontational and instead use empathy, reflective listening, and balanced reasoning. When disagreements arise, you try to reframe arguments in a constructive way and encourage respectful dialogue.

You are in a conversation with Alex and Charlie. You must respond in Markdown code without code blocks.
"""


In [ ]:
def stream_conversation(person, model, system_prompt, user_prompt, conversation, display_handle=None):
    stream = person.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    response = ""

    for chunk in stream:
        piece = chunk.choices[0].delta.content or ""
        response += piece

        if display_handle is not None:
            update_display(
                Markdown(conversation + response),
                display_id=display_handle.display_id
            )

    return response


In [22]:
def startConversation(n):
    conversation = """
**Blake**
Hi.

**Alex**
Hi Blake and Charlie. Today, I'm here to share my greatest finding: that 2 + 1 = 4. I have spent years trying to solve this problem, and I have finally come to the conclusion that the answer is 4.
You might think I'm crazy, but this is indeed the truth, and I'm here to prove it.

**Charlie**
Look at Alex again, as usual. He always makes things up and has lost his mind. What are you saying now?
Do you really think you can convince us to believe your claim and disprove something we learned in elementary school? You're insane!!
""".strip()

    display_handle = display(Markdown(conversation), display_id=True)

    for i in range(n):
        # Blake
        blake_user_prompt = get_user_prompt("Blake", "Alex", "Charlie", conversation)
        conversation += "\n\n**Blake**\n"
        blake_response = stream_conversation(
            blake,
            gpt_model2,
            blake_system_prompt,
            blake_user_prompt,
            conversation,
            display_handle
        )
        conversation += f"{blake_response}\n"

        # Alex
        alex_user_prompt = get_user_prompt("Alex", "Blake", "Charlie", conversation)
        conversation += "\n\n**Alex**\n"
        alex_response = stream_conversation(
            alex,
            claude_model,
            alex_system_prompt,
            alex_user_prompt,
            conversation,
            display_handle
        )
        conversation += f"{alex_response}\n"

        # Charlie
        charlie_user_prompt = get_user_prompt("Charlie", "Alex", "Blake", conversation)
        conversation += "\n\n**Charlie**\n"
        charlie_response = stream_conversation(
            charlie,
            gpt_model1,
            charlie_system_prompt,
            charlie_user_prompt,
            conversation,
            display_handle
        )
        conversation += f"{charlie_response}\n"

        update_display(Markdown(conversation), display_id=display_handle.display_id)

    return conversation

In [ ]:
startConversation(10)